# 🔄 Fluxos de Trabalho Básicos de Agentes com Microsoft Foundry (Python)

## 📋 Tutorial de Orquestração de Fluxo de Trabalho

Este notebook apresenta as poderosas capacidades do **Construtor de Fluxos de Trabalho** do Microsoft Agent Framework. Aprenda a criar fluxos de trabalho sofisticados e em múltiplas etapas que podem lidar com processos empresariais complexos e coordenar várias operações de IA de forma integrada.

> **Nota de migração:** Este exemplo anteriormente referenciava GitHub Models. GitHub Models está obsoleto (será desativado em julho de 2026), então agora utiliza **Microsoft Foundry** por meio do `FoundryChatClient`, que direciona para a **API de Respostas** do Azure OpenAI.

## 🎯 Objetivos de Aprendizagem

### 🏗️ **Arquitetura de Fluxo de Trabalho**
- **Construtor de Fluxo de Trabalho**: Projetar e orquestrar processos complexos em múltiplas etapas
- **Execução Orientada a Eventos**: Gerenciar eventos e transições de estado do fluxo de trabalho
- **Design Visual de Fluxo de Trabalho**: Criar e visualizar estruturas de fluxo de trabalho
- **Integração com Microsoft Foundry**: Utilizar modelos de IA dentro dos contextos de fluxo de trabalho

### 🔄 **Orquestração de Processos**
- **Operações Sequenciais**: Encadear múltiplas tarefas de agentes em ordem lógica
- **Lógica Condicional**: Implementar pontos de decisão e ramos nos fluxos de trabalho
- **Tratamento de Erros**: Recuperação robusta e resiliência do fluxo de trabalho
- **Gerenciamento de Estado**: Acompanhar e gerenciar o estado da execução do fluxo de trabalho

### 📊 **Padrões de Fluxo de Trabalho Empresarial**
- **Automação de Processos de Negócio**: Automatizar fluxos de trabalho organizacionais complexos
- **Coordenação Multi-Agentes**: Coordenar múltiplos agentes especializados
- **Execução Escalável**: Projetar fluxos de trabalho para operações em escala empresarial
- **Monitoramento e Observabilidade**: Acompanhar desempenho e resultados do fluxo de trabalho

## ⚙️ Pré-requisitos e Configuração

### 📦 **Dependências Necessárias**

Instale o Agent Framework com capacidades de fluxo de trabalho:

```bash
pip install agent-framework -U
```

### 🔑 **Configuração do Microsoft Foundry**

Faça login com o Azure CLI (`az login`) para que o `AzureCliCredential` possa autenticar, e então configure os detalhes do seu projeto Microsoft Foundry.

**Configuração do Ambiente (arquivo .env):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

### 🏢 **Casos de Uso Empresariais**

**Exemplos de Processos de Negócio:**
- **Onboarding de Clientes**: Fluxos de verificação e configuração em múltiplas etapas
- **Pipeline de Conteúdo**: Criação, revisão e publicação automatizadas de conteúdo
- **Processamento de Dados**: Fluxos de ETL com transformação alimentada por IA
- **Garantia de Qualidade**: Processos automatizados de teste e validação

**Benefícios do Fluxo de Trabalho:**
- 🎯 **Confiabilidade**: Execução determinística com recuperação de erros
- 📈 **Escalabilidade**: Suporte à automação de processos em alto volume
- 🔍 **Observabilidade**: Trilhas de auditoria completas e monitoramento
- 🔧 **Manutenibilidade**: Design visual e componentes modulares

## 🎨 Padrões de Design de Fluxo de Trabalho

### Estrutura Básica de Fluxo de Trabalho
```mermaid
graph TD
    A[Início] --> B[Tarefa do Agente 1]
    B --> C{Ponto de Decisão}
    C -->|Sucesso| D[Tarefa do Agente 2]
    C -->|Falha| E[Manipulador de Erros]
    D --> F[Fim]
    E --> F
```

**Componentes-Chave:**
- **WorkflowBuilder**: Motor principal de orquestração
- **WorkflowEvent**: Manipulação e comunicação de eventos
- **WorkflowViz**: Representação visual e depuração do fluxo de trabalho

Vamos construir seu primeiro fluxo de trabalho inteligente! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
